In [1]:
#loading connection to db
import pandas as pd
import sys
sys.path.append("../src")

from db_connection import get_engine

engine = get_engine()

In [2]:
#loading raw tables from PostgreSQL

customers = pd.read_sql("SELECT * FROM raw_customers", engine)
transactions = pd.read_sql("SELECT * FROM raw_transactions", engine)
feedback = pd.read_sql("SELECT * FROM raw_feedback", engine)

**Cleaning Customers Dataset**

In [ ]:

customers.head()
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [4]:
#standardizing column names

customers.columns = customers.columns.str.lower().str.strip()

In [5]:
#fixing data types

customers['totalcharges'] = pd.to_numeric(customers['totalcharges'], errors='coerce')

In [6]:
#handling missing values 

customers.isnull().sum()

customerid           0
gender               0
seniorcitizen        0
partner              0
dependents           0
tenure               0
phoneservice         0
multiplelines        0
internetservice      0
onlinesecurity       0
onlinebackup         0
deviceprotection     0
techsupport          0
streamingtv          0
streamingmovies      0
contract             0
paperlessbilling     0
paymentmethod        0
monthlycharges       0
totalcharges        11
churn                0
dtype: int64

In [7]:
customers['totalcharges'] = customers['totalcharges'].fillna(customers['totalcharges'].median())

In [8]:
#converting target to binary

customers['churn'] = customers['churn'].map({'Yes': 1, 'No': 0})

In [9]:
#removing duplicates

customers = customers.drop_duplicates()

**Cleaning Transactions Dataset**

In [10]:
#renaming columns
transactions.columns = transactions.columns.str.lower()

In [11]:
#checking imbalance
transactions['class'].value_counts()

class
0    284315
1       492
Name: count, dtype: int64

In [12]:
#reducing dataset for performance
transactions = transactions.sample(50000, random_state=42)

In [13]:
#removing duplicates
transactions = transactions.drop_duplicates()

**Cleaning Feedback Dataset (NLP)**

In [14]:
#renaming columns
feedback.columns = feedback.columns.str.lower()

In [15]:
#keeping useful columns
feedback = feedback[['text', 'score']]

In [16]:
#dropping missing text
feedback = feedback.dropna(subset=['text'])

In [17]:
#creating sentiment label
def sentiment(score):
    if score >= 4:
        return 1
    elif score <= 2:
        return 0
    else:
        return None

feedback['sentiment'] = feedback['score'].apply(sentiment)
feedback = feedback.dropna(subset=['sentiment'])

**Saving Clean Datasets**

In [19]:
customers.to_sql("clean_customers", engine, if_exists="replace", index=False)
transactions.to_sql("clean_transactions", engine, if_exists="replace", index=False)
feedback.to_sql("clean_feedback", engine, if_exists="replace", index=False)

814